### RAG pipeline - Data ingestion to vector db pipeline

In [50]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from typing import List, Any
import numpy as np

In [51]:
### Read all the pdfs inside the directory

def process_all_pdfs(pdf_directory):
    """process all PDF files in a directory"""

    all_documents=[]
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents=loader.load()

            #Add source information to metadata
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(pdf_files)} pages") 
        except Exception as e:
            print(f"Error:{e}")
    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data dorectory
all_pdf_doc = process_all_pdfs("../data")
          

Found 3 PDF files to process
Processing: docum.pdf
 Loaded 3 pages
Processing: report.pdf
 Loaded 3 pages
Processing: resume_latest.pdf
 Loaded 3 pages

 Total documents loaded: 48


In [52]:
all_pdf_doc

[Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2021-04-09T21:38:58+05:30', 'crossmarkdomains[2]': 'elsevier.com', 'crossmarkmajorversiondate': '2010-04-23', 'subject': 'ISCIENCE, 24 (2021) 102355. doi:10.1016/j.isci.2021.102355', 'author': 'Natalie Ann Lozano-Huntelman', 'mmc': '14', 'grabs': 'true', 'elsevierwebpdfspecifications': '7.0', 'crossmarkdomainexclusive': 'true', 'robots': 'noindex', 'moddate': '2021-04-09T21:41:22+05:30', 'doi': '10.1016/j.isci.2021.102355', 'crossmarkdomains[1]': 'sciencedirect.com', 'title': 'Hidden suppressive interactions are common in higher-order drug combinations', 'source': '..\\data\\pdf_files\\docum.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1', 'source_file': 'docum.pdf', 'file_type': 'pdf'}, page_content='iScience\nArticle\nHidden suppressive interactions are common in\nhigher-order drug combinations\nNatalie Ann\nLozano-\nHuntelman, April\nZhou, Elif Tekin, ...,\nSada Boyd, V

In [53]:
### Text splitting get intp chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

In [54]:
chunks = split_documents(all_pdf_doc)

Split 48 documents into 159 chunks

Example chunk:
Content: iScience
Article
Hidden suppressive interactions are common in
higher-order drug combinations
Natalie Ann
Lozano-
Huntelman, April
Zhou, Elif Tekin, ...,
Sada Boyd, Van M.
Savage, Pamela
Yeh
pamelayeh...
Metadata: {'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2021-04-09T21:38:58+05:30', 'crossmarkdomains[2]': 'elsevier.com', 'crossmarkmajorversiondate': '2010-04-23', 'subject': 'ISCIENCE, 24 (2021) 102355. doi:10.1016/j.isci.2021.102355', 'author': 'Natalie Ann Lozano-Huntelman', 'mmc': '14', 'grabs': 'true', 'elsevierwebpdfspecifications': '7.0', 'crossmarkdomainexclusive': 'true', 'robots': 'noindex', 'moddate': '2021-04-09T21:41:22+05:30', 'doi': '10.1016/j.isci.2021.102355', 'crossmarkdomains[1]': 'sciencedirect.com', 'title': 'Hidden suppressive interactions are common in higher-order drug combinations', 'source': '..\\data\\pdf_files\\docum.pdf', 'total_pages': 31, 'page': 0, '

### Embedding and Vector db


In [55]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [56]:
class EmbeddingManager:
    """Handles document embedding generation using Sentencetransformer"""
    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        """
        Inititalize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer"""

        try:
            print(f"Loading embedding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name} : {e}")
            raise

    def generate_embeddings(self,texts: List[str]) ->np.ndarray:
        """Generate embeddings for a list of texts
        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), mebedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generate embeddings for {len(texts)} texts..")

        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape : {embeddings.shape}")
        return embeddings

###initialize embedding manager



embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model : all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2678.12it/s]


Model loaded successfully. Embedding dimension: 384



###  Vector store

In [57]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 42


In [58]:
chunks


[Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2021-04-09T21:38:58+05:30', 'crossmarkdomains[2]': 'elsevier.com', 'crossmarkmajorversiondate': '2010-04-23', 'subject': 'ISCIENCE, 24 (2021) 102355. doi:10.1016/j.isci.2021.102355', 'author': 'Natalie Ann Lozano-Huntelman', 'mmc': '14', 'grabs': 'true', 'elsevierwebpdfspecifications': '7.0', 'crossmarkdomainexclusive': 'true', 'robots': 'noindex', 'moddate': '2021-04-09T21:41:22+05:30', 'doi': '10.1016/j.isci.2021.102355', 'crossmarkdomains[1]': 'sciencedirect.com', 'title': 'Hidden suppressive interactions are common in higher-order drug combinations', 'source': '..\\data\\pdf_files\\docum.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1', 'source_file': 'docum.pdf', 'file_type': 'pdf'}, page_content='iScience\nArticle\nHidden suppressive interactions are common in\nhigher-order drug combinations\nNatalie Ann\nLozano-\nHuntelman, April\nZhou, Elif Tekin, ...,\nSada Boyd, V

In [59]:
### convert the text to embeddings


texts = [doc.page_content for doc in chunks]


## Generate embeddings

embeddings = embedding_manager.generate_embeddings(texts)

## Store in the vector db

vectorstore.add_documents(chunks, embeddings)




Generate embeddings for 159 texts..


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches: 100%|██████████| 5/5 [00:14<00:00,  2.95s/it]


Generated embeddings with shape : (159, 384)
Adding 159 documents to vector store...
Successfully added 159 documents to vector store
Total documents in collection: 201


### Retriever Pipeline from vector store    

In [60]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever

In [ ]:

rag_retriever.retrieve("The prevalence of hidden suppression  ")

Retrieving documents for query: 'The prevalence of hidden suppression'
Top K: 5, Score threshold: 0.0
Generate embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.88it/s]

Generated embeddings with shape : (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_e3528183_43',
  'content': 'have shown that even with recent advancements and interest in suppression, one can severely underesti-\nmate the number of suppressive interactions by not considering hidden suppression.\nWhen examining hidden suppression, increasing the number of drugs in a combination also increases the\nnumber of possible lower-order combinations, thus possibly increasing the total number of combinations\nwith hidden suppression interaction. When we look at the overall percentage of combinations with hidden\nsuppression, this value steadily increases from 33% to 48%–59% as the number of drugs increases ( Figure 4).\nThis would explain the trends we see in Figure 5 for synergistic, additive, and antagonistic combinations.\nHowever, this does not offer a viable explanation for the negative correlation between the amount of hid-\nden suppression and the number of drugs in a co mbination of net and emergent suppressive\ncombinations.',
  'metadata': {'robots': 'n

### Integration vector db context pipeline with llm output


In [97]:
## Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from langchain_core.messages import HumanMessage,SystemMessage
from dotenv import load_dotenv
load_dotenv()


#Initialize Groq LLM(set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")
llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.5,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question concisely."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question. 
        Context:
        {context}

        Question: {query}

        Answer:"""
    message = [prompt.format(context=context,query=query)]
    response=llm.invoke(message)
    return response.content




In [ ]:
answer =  rag_simple("The prevalence of hidden suppression",rag_retriever,llm,top_k=5)

print(answer)

Retrieving documents for query: 'The prevalence of hidden suppression'
Top K: 5, Score threshold: 0.0
Generate embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 41.59it/s]

Generated embeddings with shape : (1, 384)
Retrieved 5 documents (after filtering)


The prevalence of hidden suppression is as follows:

- Increases from 33% to 48-59% as the number of drugs in a combination increases (Figure 4).
- Is present in a majority of higher-order combinations (Figure 4).
- Is found in all levels examined - 3-drug, 4-drug, and 5-drug combinations.
- Increases as the number of drugs in a combination increases.
- Is higher in net suppressive combinations compared to combinations that are not net suppressive (Table 2).
- Varies between different levels of drug combinations:
  - 76% in 3-drug versus 2-drug combinations.
  - 61% in 4-drug versus 2-drug combinations.
  - 60% in 4-drug versus 3-drug combinations.
  - 53% in 5-drug versus 4-drug combinations.
  - 41% in 5-drug versus 3-drug combinations.
  - 40% in 5-drug versus 2-drug combinations.
- Is present in a majority of net suppressive 5-drug combinations (80%).
- Occurs between the highest-order combination and all possible 2-drug combinations roughly 60% of the time in net suppressive 5-dru

### Enhanced RAG pipeline

In [100]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question .\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    message = [prompt.format(context=context, query=query)]
    response = llm.invoke(message)
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Net Suppression Classification ?", rag_retriever, llm, top_k=5, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:420])

Retrieving documents for query: 'Net Suppression Classification ?'
Top K: 5, Score threshold: 0.1
Generate embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.70it/s]

Generated embeddings with shape : (1, 384)
Retrieved 2 documents (after filtering)


Answer: Net Suppression Classification is defined as DA_N > 1.3.
Sources: [{'source': 'docum.pdf', 'page': 22, 'score': 0.3819173574447632, 'preview': 'Supplemental Tables  SI Table 1. Special Case Definitions, Related to Figure 3. A description of each special case definition for both net suppressive interactions and not net suppressive interactions. Net Suppression Classification (𝑫𝑨𝐍>𝟏.𝟑) Hidden Suppression Classification *𝒘𝐍𝒘𝐦𝐢𝐧\t𝐨𝐟\t𝐥𝐨𝐰𝐞𝐫\t𝐨𝐫𝐝𝐞𝐫𝐬...'}, {'source': 'docum.pdf', 'page': 8, 'score': 0.1538095474243164, 'preview': 'suppression present in that speciﬁc interaction type. The y axis is the percentage of each interaction type within the\ndesignated level of the drug combination, showing the overal l distribution of net or emergent interactions. For example,\nin (A) the net suppressive 4-drug combinations, 92% of the ...'}]
Confidence: 0.3819173574447632
Context Preview: Supplemental Tables  SI Table 1. Special Case Definitions, Related to Figure 3. A description of each spe

### Advanced RAG Pipeline

In [103]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("The prevalence of hidden suppression", top_k=5, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'The prevalence of hidden suppression'
Top K: 5, Score threshold: 0.1
Generate embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.98it/s]

Generated embeddings with shape : (1, 384)
Retrieved 2 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
have shown that even with recent advancements and interest in suppression, one can severely underesti-
mate the number of suppressive interactions by not considering hidde

n suppression.
When examining hidden suppression, increasing the number of drugs in a combination also increases the
number of possible lower-order combinations, thus possibly increasing the total number of combinations
with hidden suppression interaction. When we look at the overall percentage of combinations with hidden
suppression, this value steadily increases from 33% to 48%–59% as the number of drugs increases ( Figure 4).
This would explain the trends we see in Figure 5 for synergistic, additive, and antagonistic combinations.
However, this does not offer a viable explanation for the negative correlation between the amount of hid-
den suppression and the number of drugs in a co mbination of net and emergent suppressive
combinations.

there are a total of ten possible 2-drug combinations. In net suppressive 5-drug combinations, hidden sup-
pression occurs between the highest-order combination and all possible 2-drug combinations roughly 60%
of the time. This occurs in less than 2